# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src.ml import CommutativeCNNClassifier, CommutativeCNNConfig, LossWeightConfig, OptimizationConfig
from src.tensor_utils import load_unlabeled_tensor_dataset


In [2]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state.pt")

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(6, 8),
    spatial_kernel_size_z=(1, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(12,),
    temporal_st_kernel_sizes=(3,),
    temporal_ts_channels=(8,),
    temporal_ts_kernel_sizes=(5,),
    spatial_agg_channels=(12,),
    spatial_agg_kernel_size_z=(3,),
    spatial_agg_kernel_size_xy=(3,),
    spatial_agg_stride_z=(1,),
    spatial_agg_stride_xy=(1,),
    spatial_agg_pool_kernel_z=(1,),
    spatial_agg_pool_kernel_xy=(2,),
    spatial_agg_pool_stride_z=(1,),
    spatial_agg_pool_stride_xy=(2,),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=8,
    num_prototypes=8,
    dropout=0.5,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=75,
    learning_rate=2e-4,
    weight_decay=3e-3,
    early_stopping_patience=8,
    early_stopping_min_delta=0.0,
    scheduler_patience=3,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    consistency_weight=1.0,
    feature_weight=0.05,
    prototype_temperature=0.1,
)

In [3]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
unlabeled_dataset["tensors"].shape, unlabeled_dataset["metadata"].shape

(torch.Size([500, 20, 5, 96, 96]), (500, 7))

In [4]:
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(unlabeled_dataset["tensors"])
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path

cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trCC=train_commutative_consistency_loss
    trFA=train_feature_alignment_loss
     ep       lr       eta |      trL     trCC     trFA
001/075 2.00e-04   1:07:54 |   4.1020   4.0676   0.6886
002/075 2.00e-04   1:06:35 |   3.3194   3.2965   0.4582
003/075 2.00e-04   1:05:41 |   2.9726   2.9539   0.3749
004/075 2.00e-04   1:04:36 |   2.7237   2.7082   0.3103
005/075 2.00e-04   1:03:33 |   2.5101   2.4978   0.2458
006/075 2.00e-04   1:02:35 |   2.4100   2.3987   0.2271
007/075 2.00e-04   1:01:38 |   2.3758   2.3655   0.2062
008/075 2.00e-04   1:00:43 |   2.2315   2.2226   0.1795
009/075 2.00e-04     59:47 |   2.1559   2.1473   0.1719
010/075 2.00e-04     58:51 |   2.1201   2.1122   0.1589
011/075 2.00e-04     57:56 |   2.0855   2.0781   0.1491
012/075 2.00e-04     56:59 |   2.0617   2.0547   0.1393
013/075 2.00e-04     56:03 |   2.0283   2.0218   0.1294
014/075 2.00e-04     55:09 |   1.9972   1.

PosixPath('artifacts/pretrained_commutative_cnn/encoder_state.pt')

In [5]:
model.pretrain_history_.tail()

,epoch,train_loss,train_commutative_consistency_loss,train_feature_alignment_loss
47,48,1.859995,1.857347,0.052944
48,49,1.857847,1.855252,0.051895
49,50,1.858031,1.855411,0.052398
50,51,1.859902,1.857387,0.050299
51,52,1.857849,1.855491,0.047166
